[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day21-benchmarking-ttft-tpot-throughput.ipynb)

# Day 21 — Benchmarking: TTFT, TPOT, Throughput

**Today's big idea:** latency metrics for LLM serving are network-performance metrics wearing different clothes — TTFT is time-to-first-byte, TPOT is per-packet jitter, throughput is link bandwidth, goodput is bandwidth that arrives intact and on time. And **load is the metric**: the same server and prompt can report 150 ms or 5.15 s TTFT depending only on queue depth.

**Time:** ~60 min · **You need:** CPU only (harness + in-notebook mock server). A free T4 is needed *only* for the optional live-vLLM section at the end.


In [ ]:
# Cell 2: installs. requests/pandas/matplotlib are preinstalled on Colab;
# this is a no-op guard so the notebook runs anywhere.
import importlib, subprocess, sys
for pkg in ("requests", "pandas", "matplotlib"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("deps ready")

# Expected output:
# deps ready


## 1. Metric math on paper: from timestamps to TTFT/TPOT/E2E

The harness's core data is **per-token timestamps**: `t0` (request sent), `t_first` (first token arrived), `t_1..t_{n-1}` (each later token), `t_end` (stream closed). From these:

- `TTFT = t_first - t0` (queue + prefill + 1 decode, in that order)
- `TPOT = mean(t_{i+1} - t_i)` over output tokens
- `ITL p99 = 99th percentile of the gaps`
- `E2E = t_end - t0 = TTFT + TPOT × (n - 1)` — an identity, not an approximation.

We validate the metric functions on a synthetic trace with *known* ground truth before touching any server.


In [ ]:
# Cell 4: metric functions + ground-truth validation (no server involved)
import statistics

def percentile(xs, p):
    xs = sorted(xs)
    k = (len(xs) - 1) * p / 100
    f, c = int(k), min(int(k) + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)

def compute_metrics(t0, token_times, t_end):
    """token_times: arrival times of the n output tokens, in order."""
    n = len(token_times)
    ttft = token_times[0] - t0
    gaps = [b - a for a, b in zip(token_times, token_times[1:])]
    tpot = statistics.fmean(gaps) if gaps else 0.0
    return {
        "ttft_ms": ttft * 1000,
        "tpot_ms": tpot * 1000,
        "itl_p99_ms": percentile(gaps, 99) * 1000 if gaps else 0.0,
        "e2e_ms": (t_end - t0) * 1000,
        "n_tokens": n,
    }

# Ground truth: TTFT=120ms, TPOT=30ms, n=32 tokens
ttft, tpot, n = 0.120, 0.030, 32
token_times = [ttft + i * tpot for i in range(n)]
t_end = token_times[-1] + 0.005  # stream teardown
m = compute_metrics(0.0, token_times, t_end)

assert abs(m["ttft_ms"] - 120) < 1e-9, m
assert abs(m["tpot_ms"] - 30) < 1e-9, m
assert abs(m["itl_p99_ms"] - 30) < 1e-9, m
# The E2E identity: E2E ~= TTFT + TPOT*(n-1)
lhs = m["e2e_ms"] - 5.0          # remove teardown
rhs = 120 + 30 * (n - 1)
assert abs(lhs - rhs) < 1e-6, (lhs, rhs)
print(f"ground truth OK: ttft={m['ttft_ms']:.1f}ms tpot={m['tpot_ms']:.1f}ms "
      f"itl_p99={m['itl_p99_ms']:.1f}ms e2e={m['e2e_ms']:.1f}ms")

# Expected output:
# ground truth OK: ttft=120.0ms tpot=30.0ms itl_p99=30.0ms e2e=1055.0ms


## 2. `bench.py`: the reusable harness

This is the permanent tool — you'll run it on every serving day from here to Day 90. It takes a server URL, request count, prompt/output lengths, and concurrency; streams SSE responses; records per-token timestamps; writes one JSON line per request (workload args included, so runs are repeatable); and prints p50/p99 summaries.


In [ ]:
# Cell 6: write the reusable harness to disk
%%writefile bench.py
"""bench.py -- minimal LLM serving benchmark harness (Day 21).

Streams SSE completions, records per-token timestamps, computes
TTFT / TPOT / ITL-p99 / E2E per request, writes JSONL, prints p50/p99.
"""
import argparse, concurrent.futures, json, statistics, sys, time
import requests

def percentile(xs, p):
    xs = sorted(xs)
    k = (len(xs) - 1) * p / 100
    f, c = int(k), min(int(k) + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)

def one_request(url, prompt_len, out_len, timeout=120):
    prompt = "token " * prompt_len
    t0 = time.time()
    token_times, n_tok = [], 0
    r = requests.post(url, json={"prompt": prompt, "max_tokens": out_len,
                                "stream": True}, stream=True, timeout=timeout)
    r.raise_for_status()
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data:"):
            continue
        data = line[5:].strip()
        if data == "[DONE]":
            break
        token_times.append(time.time())
        n_tok += 1
    t_end = time.time()
    gaps = [b - a for a, b in zip(token_times, token_times[1:])]
    return {
        "ttft_ms": (token_times[0] - t0) * 1000 if token_times else None,
        "tpot_ms": statistics.fmean(gaps) * 1000 if gaps else None,
        "itl_p99_ms": percentile(gaps, 99) * 1000 if gaps else None,
        "e2e_ms": (t_end - t0) * 1000,
        "n_tokens": n_tok,
        "prompt_len": prompt_len, "out_len": out_len,
    }

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", required=True)
    ap.add_argument("--n-requests", type=int, default=20)
    ap.add_argument("--prompt-len", type=int, default=64)
    ap.add_argument("--out-len", type=int, default=32)
    ap.add_argument("--concurrency", type=int, default=1)
    ap.add_argument("--jsonl", default="bench.jsonl")
    a = ap.parse_args()

    t_start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=a.concurrency) as ex:
        futs = [ex.submit(one_request, a.url, a.prompt_len, a.out_len)
                for _ in range(a.n_requests)]
        # warmup request excluded from stats (kernel autotune / cache warmup)
        warm = futs.pop(0).result()
        recs = [f.result() for f in futs]
    wall = time.time() - t_start  # true wall clock: submit of first -> completion of last
    with open(a.jsonl, "w") as f:
        for r in recs:
            f.write(json.dumps(r) + "\n")
    tok = sum(r["n_tokens"] for r in recs)
    print(f"requests={len(recs)} (+1 warmup)  total_tokens={tok}  wall_s={wall:.1f}")
    for k in ("ttft_ms", "tpot_ms", "itl_p99_ms", "e2e_ms"):
        xs = [r[k] for r in recs if r[k] is not None]
        print(f"{k:10s} p50={percentile(xs,50):8.1f}  p99={percentile(xs,99):8.1f}")
    print(f"throughput ~= {tok / wall:.0f} tok/s (output tokens / wall clock)")

if __name__ == "__main__":
    main()


## 3. Mock SSE server: a server with a known service rate

To find a saturation knee we need a server that *saturates*. The mock below simulates a real serving engine's key property: a fixed **service capacity** (like the iteration token budget). A semaphore with `capacity` slots admits requests; extras queue, and their TTFT grows with queue wait — Little's Law, live. Service time per request ≈ `prefill_ms + out_len × tpot_ms`.


In [ ]:
# Cell 8: threaded mock SSE server with a saturation semaphore
import json as _json, threading, time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

class MockHandler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"  # enables chunked transfer encoding
    # service model: 4 concurrent slots; prefill 100ms; 30ms per token
    capacity = threading.Semaphore(4)
    prefill_ms, tpot_ms = 100, 30
    def log_message(self, *a): pass
    def _sse(self, payload: bytes):
        # chunked framing: each SSE event is its own chunk, so the client
        # (requests -> http.client read(512)) delivers it immediately.
        # Without this, HTTP/1.0 close-delimited responses are only
        # delivered at EOF and per-token timestamps collapse.
        self.wfile.write(f"{len(payload):X}\r\n".encode() + payload + b"\r\n")
        self.wfile.flush()
    def do_POST(self):
        n = int(_json.loads(self.rfile.read(int(self.headers["Content-Length"])))["max_tokens"])
        with MockHandler.capacity:                       # admission: extras QUEUE here
            self.send_response(200)
            self.send_header("Content-Type", "text/event-stream")
            self.send_header("Transfer-Encoding", "chunked")
            self.end_headers()
            time.sleep(MockHandler.prefill_ms / 1000)    # prefill phase
            for _ in range(n):
                time.sleep(MockHandler.tpot_ms / 1000)   # decode phases
                chunk = _json.dumps({"choices": [{"delta": {"content": "tok"}}]})
                self._sse(f"data: {chunk}\n\n".encode())
            self._sse(b"data: [DONE]\n\n")
            self.wfile.write(b"0\r\n\r\n")            # end-of-chunks
            self.wfile.flush()

server = ThreadingHTTPServer(("127.0.0.1", 8123), MockHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("mock server on http://127.0.0.1:8123/v1/completions "
      "(chunked SSE, capacity=4, prefill=100ms, tpot=30ms)")

# Expected output:
# mock server on http://127.0.0.1:8123/v1/completions (chunked SSE, capacity=4, prefill=100ms, tpot=30ms)



## 4. Run the harness: concurrency 1, then the sweep

First a low-load run to calibrate: at concurrency 1 there is no queueing, so TTFT ≈ prefill (100 ms) + one decode (30 ms) ≈ 130 ms, and TPOT ≈ 30 ms.


In [ ]:
# Cell 10: harness at concurrency 1 against the mock
import subprocess, json
r = subprocess.run(
    ["python3", "bench.py", "--url", "http://127.0.0.1:8123/v1/completions",
     "--n-requests", "21", "--prompt-len", "64", "--out-len", "32",
     "--concurrency", "1", "--jsonl", "bench_c1.jsonl"],
    capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, r.stderr
recs = [json.loads(l) for l in open("bench_c1.jsonl")]
ttft_p50 = sorted(x["ttft_ms"] for x in recs)[len(recs)//2]
tpot_p50 = sorted(x["tpot_ms"] for x in recs)[len(recs)//2]
assert 90 < ttft_p50 < 180, ttft_p50   # ~130ms: prefill 100 + one decode 30
assert 20 < tpot_p50 < 45, tpot_p50    # ~30ms programmed rate
print(f"calibration OK: ttft_p50={ttft_p50:.0f}ms (expect ~130), "
      f"tpot_p50={tpot_p50:.0f}ms (expect ~30)")

# Expected output (timing jitter ±10ms is normal; measured 2026-09-29):
# requests=20 (+1 warmup)  total_tokens=640  wall_s=22.5
# ttft_ms    p50=   132.2  p99=   140.5
# tpot_ms    p50=    30.2  p99=    30.4
# itl_p99_ms p50=    30.8  p99=    34.1
# e2e_ms     p50=  1069.4  p99=  1080.6
# throughput ~= 28 tok/s (output tokens / wall clock)
# calibration OK: ttft_p50=132ms (expect ~130), tpot_p50=30ms (expect ~30)


## 5. The sweep: find the saturation knee

Now concurrencies {1, 4, 16}. Watch two curves: throughput (should rise then plateau) and TTFT p99 (should stay flat then climb). The knee is where TTFT p99 first exceeds 2× its low-load value — the point where queue wait, not prefill, starts dominating TTFT.


In [ ]:
# Cell 12: sweep {1, 4, 16} and plot the trade-off curve
import subprocess, json
import matplotlib.pyplot as plt

def pct(xs, p):
    xs = sorted(xs); k = (len(xs)-1)*p/100
    f, c = int(k), min(int(k)+1, len(xs)-1)
    return xs[f] + (xs[c]-xs[f])*(k-f)

rows = []
for conc in (1, 4, 16):
    r = subprocess.run(
        ["python3", "bench.py", "--url", "http://127.0.0.1:8123/v1/completions",
         "--n-requests", "33", "--prompt-len", "64", "--out-len", "32",
         "--concurrency", str(conc), "--jsonl", f"bench_c{conc}.jsonl"],
        capture_output=True, text=True, check=True)
    # true wall clock comes from the harness (first submit -> last completion)
    wall = float([l for l in r.stdout.splitlines() if "wall_s=" in l][0]
                 .split("wall_s=")[1].split()[0])
    recs = [json.loads(l) for l in open(f"bench_c{conc}.jsonl")]
    ttfts = [x["ttft_ms"] for x in recs]; tpots = [x["tpot_ms"] for x in recs]
    tok = sum(x["n_tokens"] for x in recs)
    rows.append({"conc": conc, "wall": wall, "tput": tok / wall,
                 "ttft_p99": pct(ttfts, 99), "tpot_p99": pct(tpots, 99),
                 "recs": recs})
    print(f"c={conc:2d}  throughput={tok/wall:6.0f} tok/s  "
          f"ttft_p99={pct(ttfts,99):7.0f} ms  tpot_p99={pct(tpots,99):5.0f} ms")

base = rows[0]["ttft_p99"]
knee = next((r["conc"] for r in rows if r["ttft_p99"] > 2 * base), None)
print(f"\nlow-load ttft_p99={base:.0f} ms -> 2x rule fires at concurrency ~{knee}; "
      f"the economic knee (throughput stops rising) is at 4 = the mock's capacity")

fig, ax1 = plt.subplots(figsize=(7, 4))
cs = [r["conc"] for r in rows]
ax1.plot(cs, [r["tput"] for r in rows], "o-", label="throughput (tok/s)")
ax1.set_xlabel("concurrency"); ax1.set_ylabel("throughput (tok/s)")
ax2 = ax1.twinx()
ax2.plot(cs, [r["ttft_p99"] for r in rows], "s--", color="C1",
         label="TTFT p99 (ms)")
ax2.set_ylabel("TTFT p99 (ms)", color="C1")
ax1.set_title("Throughput-latency trade-off: the saturation knee")
fig.tight_layout(); plt.show()

# Expected output (measured 2026-09-29, 4 runs; throughput stable, tail varies):
# c= 1  throughput=    29 tok/s  ttft_p99=    139 ms  tpot_p99=   31 ms
# c= 4  throughput=   105 tok/s  ttft_p99=    144 ms  tpot_p99=   31 ms
# c=16  throughput=   106 tok/s  ttft_p99=  3300-6500 ms  tpot_p99= 30 ms
# low-load ttft_p99=139 ms -> 2x rule fires at concurrency ~16; the economic
# knee (throughput stops rising) is at 4 = the mock's capacity.
# (The c=16 tail varies run to run: semaphore wakeup order is not
# deterministic. What matters: p99 >> 2x low-load while throughput is flat.)


## 6. Goodput: throughput with an SLO filter

Raw throughput at concurrency 16 looks best — but apply the chat SLO (`TTFT < 1000 ms` and per-request `itl_p99 < 100 ms`) and count only tokens from requests that pass. Also verify the E2E identity on a real measured request.


In [ ]:
# Cell 14: goodput vs throughput, and the E2E identity check
def goodput(rows, ttft_slo=1000, itl_slo=100):
    out = {}
    for r in rows:
        good = [x for x in r["recs"]
                if x["ttft_ms"] < ttft_slo and x["itl_p99_ms"] < itl_slo]
        wall = r["wall"]  # true wall clock, parsed from the harness output
        out[r["conc"]] = (sum(x["n_tokens"] for x in good) / wall,
                          sum(x["n_tokens"] for x in r["recs"]) / wall)
    return out

print("SLO: ttft < 1000ms, itl_p99 < 100ms")
for conc, (gp, tp) in goodput(rows).items():
    print(f"c={conc:2d}  throughput={tp:6.0f} tok/s  goodput={gp:6.0f} tok/s"
          f"  waste={(1-gp/tp)*100:4.1f}%")

# E2E identity on a measured request (definition, not approximation)
x = rows[0]["recs"][0]
lhs = x["e2e_ms"]; rhs = x["ttft_ms"] + x["tpot_ms"] * (x["n_tokens"] - 1)
assert abs(lhs - rhs) / lhs < 0.02, (lhs, rhs)  # <2%: stream teardown slack
print(f"\nE2E identity OK: measured={lhs:.0f}ms  ttft+tpot*(n-1)={rhs:.0f}ms")

# Expected output (measured 2026-09-29):
# SLO: ttft < 1000ms, itl_p99 < 100ms
# c= 1  throughput=    29 tok/s  goodput=    29 tok/s  waste= 0.0%
# c= 4  throughput=   104 tok/s  goodput=   104 tok/s  waste= 0.0%
# c=16  throughput=   107 tok/s  goodput=    13 tok/s  waste=87.5%
# E2E identity OK: measured=1076ms  ttft+tpot*(n-1)=1076ms


## 7. Optional: the same sweep against real vLLM (T4 only)

This cell runs only where `torch.cuda.is_available()` is true. It installs vLLM (~2 GB), serves `gpt2`, and points the same `bench.py` at it. On CPU it prints the exact commands to run instead — nothing fails.


In [ ]:
# Cell 16: live vLLM sweep -- T4 only, guarded
import torch, subprocess, time, os

if not torch.cuda.is_available():
    print("No GPU here -- CPU path done. On a T4 Colab runtime, run:")
    print("  !pip install -q vllm")
    print("  !python -m vllm.entrypoints.openai.api_server --model gpt2 &")
    print('  !python3 bench.py --url http://localhost:8000/v1/completions '
          '--n-requests 33 --prompt-len 64 --out-len 32 --concurrency 4')
    print("then repeat with --concurrency 1 and 16, and find the knee "
          "exactly as in section 5.")
else:
    subprocess.run(["pip", "install", "-q", "vllm"], check=True)
    srv = subprocess.Popen(
        ["python", "-m", "vllm.entrypoints.openai.api_server",
         "--model", "gpt2", "--gpu-memory-utilization", "0.7"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(60)  # model load
    try:
        for conc in (1, 4, 16):
            r = subprocess.run(
                ["python3", "bench.py", "--url",
                 "http://localhost:8000/v1/completions",
                 "--n-requests", "33", "--prompt-len", "64", "--out-len", "32",
                 "--concurrency", str(conc), "--jsonl", f"vllm_c{conc}.jsonl"],
                capture_output=True, text=True)
            print(f"--- concurrency {conc} ---\n{r.stdout}")
    finally:
        srv.terminate()
    print("Record in your notes: 'T4 + gpt2 saturates at ~N concurrent'.")

# Expected output (CPU):
# No GPU here -- CPU path done. On a T4 Colab runtime, run:
#   ...


## 8. Checkpoints (self-check) + record your numbers

1. **Request: TTFT = 150 ms, TPOT = 40 ms, 100 output tokens. E2E?** → 150 + 99 × 40 = 4,110 ms ≈ 4.1 s. (The first token's time is inside TTFT: it's *n − 1* gaps.)
2. **Goodput vs throughput, one sentence each?** → Throughput counts all tokens; goodput counts only tokens from requests that met their SLOs.
3. **Streaming improves which metric, and which are unchanged?** → Perceived TTFT improves; TPOT and throughput are unchanged.
4. **Throughput rises with concurrency but goodput falls. Regime and fix?** → Past the saturation knee: queue wait pushes requests past SLO. Back concurrency down to the knee.
5. **Why per-token timestamps, not averages?** → Timestamps let you recompute any percentile or SLO filter later; averages destroy the distribution.

**Record in your notes** (the "What to measure" table in the PDF): mock knee concurrency, goodput-at-knee vs raw-throughput-at-16, and — if you ran section 7 — `T4 + gpt2 saturates at ~N concurrent`. This harness (`bench.py`) is yours now: commit it to `week03/bench.py` in the repo.

**Tomorrow (Day 22 — Load Testing + SLOs):** today you built the speedometer; tomorrow you write the speed limit — arrival patterns, product-matched SLOs, and the load test that proves or breaks them.
